# DeepHit benchmarks



In [1]:
import os
from pathlib import Path
import sys
node_type = os.getenv('BB_CPU')
# venv_dir = f'/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-{node_type}'
venv_dir = "/rds/projects/s/subramaa-mum-predict/CharlesGadd_Oxford/virtual_envs/SurvivEHR-3.10.4"
venv_site_pkgs = Path(venv_dir) / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
if venv_site_pkgs.exists():
    sys.path.insert(0, str(venv_site_pkgs))
    print(f"Added path '{venv_site_pkgs}' at start of search paths.")
else:
    print(f"Path '{venv_site_pkgs}' not found. Check that it exists and/or that it exists for node-type '{node_type}'.")

%load_ext autoreload
%autoreload 2

Added path '/rds/projects/s/subramaa-mum-predict/CharlesGadd_Oxford/virtual_envs/SurvivEHR-3.10.4/lib/python3.10/site-packages' at start of search paths.


In [2]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logging
from tqdm import tqdm
from hydra import compose, initialize
from omegaconf import OmegaConf

from SurvivEHR.examples.modelling.benchmarks.make_method_loaders import get_dataloaders
from SurvivEHR.examples.modelling.benchmarks.DeepHit.train_deephit import run_experiment


torch.manual_seed(1337)
logging.basicConfig(level=logging.INFO)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = "cpu"    # if more informative debugging statements are needed
print(f"Using device: {device}.")

Using device: cpu.


In [4]:
# Data and model settings for ablation and full model study 

# dataset = "CVD"
# competing_risk = True
# sample_sizes = [None] + [int(np.exp(_log_n)) for _log_n in np.linspace(np.log(3000), np.log(500000), 10)]
# seeds = [1,2,3,4,5]

# dataset_path = f"/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/FineTune_{dataset}/benchmark_data/"
# ckpt_path_with_prefix = f"outputs/{dataset}/"

In [6]:
# Data and model settings for regional study 

dataset = "CVD"

if dataset == "Hypertension":
    competing_risk = False
    sample_sizes = [None]
    seeds = [1, 2, 3, 4, 5]
    
elif dataset == "CVD":
    competing_risk = False
    sample_sizes = [None]
    seeds = [1, 2, 3, 4, 5]
    
elif dataset == "MM":
    competing_risk = False
    sample_sizes = [20000]
    seeds = [1]
    
else:
    raise NotImplementedError

dataset_path = f"/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/{dataset}_North East/benchmark_data/"
ckpt_path_with_prefix = f"outputs/{dataset}/regional_"

In [ ]:
bins=200

model_names = []
all_ctd = []
all_ibs = []
all_inbll = []
for sample_size in sample_sizes:

    seed_model_names = []
    seed_ctd = []
    seed_ibs = []
    seed_inbll = []
    for seed in seeds:

        model_name = f"DeepHit-{dataset}-Ns{sample_size}-seed{seed}"
        torch.manual_seed(seed)

        # Load data
        dataset_train, dataset_val, dataset_test, meta_information = get_dataloaders(dataset_path, competing_risk, "deephit", sample_size=sample_size, seed=seed, bins=bins)
        print(dataset_train)
        print(len(dataset_val))
        print(len(dataset_test))

        # Train benchmark
        result_dict = run_experiment(dataset_train, dataset_val, dataset_test, meta_information)
        print(result_dict)
    
        # Record
        seed_model_names.append(model_name)
        seed_ctd.append(result_dict["ctd"])
        seed_ibs.append(result_dict["ibs"])
        seed_inbll.append(result_dict["inbll"])

    # Record
    model_names.append(seed_model_names)
    all_ctd.append(seed_ctd)
    all_ibs.append(seed_ibs)
    all_inbll.append(seed_inbll)
 

Loading training dataset from /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/CVD_North East/benchmark_data/all.pickle
Loading validation/test datasets from /rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/CVD_North East/benchmark_data/all.pickle
2
2
2
lr_finder best lr: 0.04229242874389523
setting to lr: 0.01
0:	[1s / 1s],		train_loss: 0.2934,	val_loss: 0.2847
1:	[2s / 4s],		train_loss: 0.2768,	val_loss: 0.2841
2:	[7s / 12s],		train_loss: 0.2733,	val_loss: 0.2824


In [ ]:
print(model_names)
print(f"{np.mean(all_ctd)} +- {np.std(all_ctd)}")
print(f"{np.mean(all_ibs)} +- {np.std(all_ibs)}")
print(f"{np.mean(all_inbll)} +- {np.std(all_inbll)}")
